# Оживление картинки петлёй (Wan 2.1 Fun 1.3B)

Делает короткую бесшовную петлю: двигается только выбранная область (пар, гирлянда, страница), всё остальное — исходная картинка без изменений.

**Как запускать:** Среда выполнения → Сменить среду → **T4 GPU**. Потом по очереди каждую ячейку (▶). Первый запуск долгий: установка и скачивание модели (~10–15 мин).

⚠️ Блокнот ещё не проверен на живой видеокарте — первый запуск тестовый.

In [ ]:
#@title 1. Настройки
# Картинка на Google Диске (путь внутри «Мой диск»). Пусто — возьмётся самая новая .png/.jpg из папки ПАПКА.
IMAGE = ""  #@param {type:"string"}
ПАПКА = "Оживление"  #@param {type:"string"}
# Что движется — по-английски, коротко. Камера неподвижна.
PROMPT = "steam gently rising from the cup, static camera, still illustration, nothing else moves"  #@param {type:"string"}
# Область движения в процентах кадра: слева, сверху, ширина, высота. "0,0,100,100" — весь кадр.
ОБЛАСТЬ = "40,30,25,30"  #@param {type:"string"}
СЕКУНД = 3  #@param {type:"slider", min:2, max:5, step:1}
SEED = 42  #@param {type:"integer"}


In [ ]:
#@title 2. Диск и установка (один раз за сеанс)
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
if not os.path.exists('/content/VideoX-Fun'):
    subprocess.run('git clone --depth 1 https://github.com/aigc-apps/VideoX-Fun /content/VideoX-Fun', shell=True, check=True)
subprocess.run('pip -q install -r /content/VideoX-Fun/requirements.txt', shell=True)
subprocess.run('pip -q install huggingface_hub', shell=True)
print('готово')


In [ ]:
#@title 3. Модель (один раз за сеанс, ~7 ГБ)
from huggingface_hub import snapshot_download
MODEL = '/content/VideoX-Fun/models/Diffusion_Transformer/Wan2.1-Fun-1.3B-InP'
snapshot_download('alibaba-pai/Wan2.1-Fun-1.3B-InP', local_dir=MODEL)
print('модель на месте')


In [ ]:
#@title 4. Готовим картинку
import glob
from PIL import Image
base = '/content/drive/MyDrive'
if IMAGE.strip():
    src = os.path.join(base, IMAGE.strip())
else:
    files = [f for e in ('png','jpg','jpeg') for f in glob.glob(f'{base}/{ПАПКА}/*.{e}')]
    assert files, f'Нет картинок в папке «{ПАПКА}» на Диске'
    src = max(files, key=os.path.getmtime)
orig = Image.open(src).convert('RGB')
W, H = orig.size
# размер для модели: ~832 по длинной стороне, кратно 16
scale = 832 / max(W, H)
mw, mh = int(W*scale)//16*16, int(H*scale)//16*16
orig.resize((mw, mh), Image.LANCZOS).save('/content/start.png')
print('картинка:', src, (W, H), '→ модель', (mw, mh))


In [ ]:
#@title 5. Оживляем (на T4 это долго — ждём)
import re
script = '/content/VideoX-Fun/examples/wan2.1_fun/predict_i2v.py'
s = open(script).read()
fps = 16
frames = СЕКУНД * fps + 1
vals = {
    'GPU_memory_mode': '"model_cpu_offload_and_qfloat8"',
    'model_name': repr(MODEL),
    'sample_size': f'[{mh}, {mw}]',
    'video_length': str(frames),
    'fps': str(fps),
    'weight_dtype': 'torch.float16',   # T4 не умеет bfloat16
    'prompt': repr(PROMPT),
    'negative_prompt': repr('camera movement, zoom, pan, morphing, distortion, blurry, deformed character, text changes'),
    'validation_image_start': repr('/content/start.png'),
    'validation_image_end': repr('/content/start.png'),   # тот же кадр в конце — бесшовная петля
    'seed': str(SEED),
    'save_path': repr('/content/out'),
}
missing = []
for k, v in vals.items():
    s2, n = re.subn(rf'^{k}\s*=.*$', f'{k} = {v}', s, count=1, flags=re.M)
    if n: s = s2
    else: missing.append(k)
open('/content/predict_loop.py', 'w').write(s)
if missing: print('⚠️ не нашла в скрипте:', missing, '— пришли это Клоду')
r = subprocess.run('cd /content/VideoX-Fun && python /content/predict_loop.py', shell=True)
vids = sorted(glob.glob('/content/out/**/*.mp4', recursive=True), key=os.path.getmtime)
assert vids, 'Видео не появилось — пришли Клоду последние строки вывода'
raw = vids[-1]; print('готово:', raw)


In [ ]:
#@title 6. Двигается только область, остальное — исходная картинка. Сохраняем на Диск
from PIL import ImageFilter
import shutil
x, y, w, h = [float(v) for v in ОБЛАСТЬ.split(',')]
mask = Image.new('L', (W, H), 0)
box = (int(W*x/100), int(H*y/100), int(W*(x+w)/100), int(H*(y+h)/100))
mask.paste(255, box)
feather = max(8, int(min(box[2]-box[0], box[3]-box[1]) * 0.15))
mask = mask.filter(ImageFilter.GaussianBlur(feather))
os.makedirs('/content/fr', exist_ok=True); os.makedirs('/content/comp', exist_ok=True)
for f in glob.glob('/content/fr/*') + glob.glob('/content/comp/*'): os.remove(f)
subprocess.run(f'ffmpeg -loglevel error -i "{raw}" /content/fr/%04d.png', shell=True, check=True)
for f in sorted(glob.glob('/content/fr/*.png')):
    fr = Image.open(f).convert('RGB').resize((W, H), Image.LANCZOS)
    Image.composite(fr, orig, mask).save(f.replace('/fr/', '/comp/'))
name = os.path.splitext(os.path.basename(src))[0]
out = f'/content/{name} (петля).mp4'
subprocess.run(f'ffmpeg -loglevel error -y -framerate {fps} -i /content/comp/%04d.png -c:v libx264 -pix_fmt yuv420p -crf 16 "{out}"', shell=True, check=True)
dst = f'{base}/{ПАПКА}'; os.makedirs(dst, exist_ok=True)
shutil.copy(out, dst); shutil.copy(raw, f'{dst}/{name} (сырое).mp4')
print('на Диске:', dst)
from IPython.display import Video
Video(out, embed=True, width=640)
